In [ ]:
%%writefile build_temporal_graph.py
import numpy as np
import pandas as pd

FEATURES_PATH = "sample_features.csv"
EDGES_PATH = "sample_edges.csv"

ALPHA, BETA, GAMMA = 1.0, 1.0, 1.0
TYPE_WEIGHT = {"reply": ALPHA, "mention": BETA, "follow": GAMMA}
TYPE_CODE = {"reply": 0, "mention": 1, "follow": 2} 


def load_node_features():
    df = pd.read_csv(FEATURES_PATH)
    feature_cols = [c for c in df.columns if c.startswith(("uf_", "cf_", "nf_", "tf_"))]
    raw = df[feature_cols].astype(float)

    if raw.isna().any().any():
        raw = raw.fillna(raw.mean())

    mean = raw.mean()
    std = raw.std().replace(0, 1.0)
    normalized = (raw - mean) / std

    x = normalized.values.astype(np.float32)
    y = (df["label"].str.lower() == "bot").astype(int).values
    ids = df["id"].values
    id_to_idx = {uid: i for i, uid in enumerate(ids)}

    return x, y, ids, id_to_idx


def build_temporal_edges(id_to_idx):
    edges = pd.read_csv(EDGES_PATH)

    before = len(edges)
    edges = edges[edges["source"].isin(id_to_idx) & edges["target"].isin(id_to_idx)]
    dropped = before - len(edges)
    if dropped:
        print(f"WARNING: dropped {dropped} edges referencing ids outside the feature set")

    unmapped = set(edges["edge_type"].unique()) - set(TYPE_WEIGHT)
    if unmapped:
        print(f"WARNING: unmapped edge_type values (given weight 1.0, code -1): {unmapped}")

    src = edges["source"].map(id_to_idx).values
    dst = edges["target"].map(id_to_idx).values
    timestamp = edges["timestamp"].values.astype(np.int64)
    weight = edges["edge_type"].map(TYPE_WEIGHT).fillna(1.0).values.astype(np.float32)
    edge_type_code = edges["edge_type"].map(TYPE_CODE).fillna(-1).astype(int).values

    order = np.argsort(timestamp, kind="stable")
    return (src[order], dst[order], timestamp[order],
            weight[order], edge_type_code[order])


class NeighborFinder:


    def __init__(self, num_nodes, src, dst, timestamp, weight, edge_type_code):
        # adjacency[node] = list of (neighbor, edge_idx, timestamp), built
        # from both directions since an interaction connects both parties.
        adjacency = [[] for _ in range(num_nodes)]
        for e_idx in range(len(src)):
            s, d, t = src[e_idx], dst[e_idx], timestamp[e_idx]
            adjacency[s].append((d, e_idx, t))
            adjacency[d].append((s, e_idx, t))

        # Sort each node's own neighbor list by time, then split into
        # parallel numpy arrays for fast searchsorted lookups.
        self.node_neighbors = []
        self.node_edge_idxs = []
        self.node_timestamps = []
        for node in range(num_nodes):
            entries = sorted(adjacency[node], key=lambda x: x[2])
            self.node_neighbors.append(np.array([e[0] for e in entries], dtype=np.int64))
            self.node_edge_idxs.append(np.array([e[1] for e in entries], dtype=np.int64))
            self.node_timestamps.append(np.array([e[2] for e in entries], dtype=np.int64))

        self.weight = weight
        self.edge_type_code = edge_type_code

    def get_temporal_neighbors(self, node, cutoff_time, n_neighbors=20):
     
        ts_array = self.node_timestamps[node]
        # searchsorted finds the insertion point for cutoff_time, which is
        # exactly the count of neighbors strictly before it (since ts_array
        # is sorted ascending).
        cutoff_idx = np.searchsorted(ts_array, cutoff_time, side="left")

        neighbors = self.node_neighbors[node][:cutoff_idx]
        edge_idxs = self.node_edge_idxs[node][:cutoff_idx]
        timestamps = ts_array[:cutoff_idx]

        # Keep only the most recent n_neighbors (closest in time to the
        # query) - this is the standard TGAT sampling strategy, since
        # recent interactions are assumed most informative and it bounds
        # the attention computation to a fixed size regardless of degree.
        if len(neighbors) > n_neighbors:
            neighbors = neighbors[-n_neighbors:]
            edge_idxs = edge_idxs[-n_neighbors:]
            timestamps = timestamps[-n_neighbors:]

        return neighbors, edge_idxs, timestamps


def main():
    x, y, ids, id_to_idx = load_node_features()
    src, dst, timestamp, weight, edge_type_code = build_temporal_edges(id_to_idx)

    print(f"Node features: {x.shape}")
    print(f"Temporal events: {len(src)} (sorted {pd.to_datetime(timestamp.min(), unit='s').date()} "
          f"to {pd.to_datetime(timestamp.max(), unit='s').date()})")

    finder = NeighborFinder(len(ids), src, dst, timestamp, weight, edge_type_code)

  
    busiest_node = int(np.argmax([len(t) for t in finder.node_timestamps]))
    all_ts = finder.node_timestamps[busiest_node]
    if len(all_ts) >= 2:
        mid_time = int(all_ts[len(all_ts) // 2])
        neighbors, edge_idxs, ts = finder.get_temporal_neighbors(busiest_node, mid_time, n_neighbors=20)
        print(f"\nSanity check on node {ids[busiest_node]} ({len(all_ts)} total events):")
        print(f"  Query time: {pd.to_datetime(mid_time, unit='s')}")
        print(f"  Found {len(neighbors)} temporal neighbors before this time")
        assert (ts < mid_time).all(), "LEAK: found a neighbor at or after the query time"
        print(f"  PASS: all {len(ts)} returned neighbors are strictly before the query time")

    
        earlier_time = int(all_ts[max(0, len(all_ts) // 4)])
        n_mid_uncapped, _, _ = finder.get_temporal_neighbors(busiest_node, mid_time, n_neighbors=10_000)
        n_early_uncapped, _, _ = finder.get_temporal_neighbors(busiest_node, earlier_time, n_neighbors=10_000)
        assert len(n_early_uncapped) < len(n_mid_uncapped), \
            "querying an earlier time did not strictly reduce the neighbor count"
        brute_force = sum(1 for i in range(len(src))
                           if (src[i] == busiest_node or dst[i] == busiest_node) and timestamp[i] < mid_time)
        assert brute_force == len(n_mid_uncapped), \
            f"NeighborFinder ({len(n_mid_uncapped)}) disagrees with brute-force count ({brute_force})"
        print(f"  PASS: earlier time strictly reduces neighbor count when uncapped "
              f"({len(n_early_uncapped)} < {len(n_mid_uncapped)}), and matches brute-force "
              f"ground truth exactly ({brute_force})")

    np.savez("node_features.npz", x=x, y=y, ids=ids)
    np.savez("temporal_edges.npz", src=src, dst=dst, timestamp=timestamp,
             weight=weight, edge_type_code=edge_type_code)
    print("\nSaved node_features.npz and temporal_edges.npz")


#if __name__ == "__main__":
 #   main()


Writing build_temporal_graph.py


In [ ]:
import numpy as np
import torch
import torch.nn as nn
import torch.nn.functional as F
import pandas as pd
import os

from build_temporal_graph import NeighborFinder

NODE_FEATURES_PATH = "/kaggle/input/datasets/judithannbenny/tnsgat-data/node_features.npz"
TEMPORAL_EDGES_PATH = "/kaggle/input/datasets/judithannbenny/tnsgat-data/temporal_edges.npz"
node_dir = NODE_FEATURES_PATH  # already points at the "directory"
x = np.load(os.path.join(node_dir, "x.npy"))
y = np.load(os.path.join(node_dir, "y.npy"))
ids = np.load(os.path.join(node_dir, "ids.npy"), allow_pickle=True)
edge_dir = TEMPORAL_EDGES_PATH
src = np.load(os.path.join(edge_dir, "src.npy"))
dst = np.load(os.path.join(edge_dir, "dst.npy"))
timestamp = np.load(os.path.join(edge_dir, "timestamp.npy"))
weight = np.load(os.path.join(edge_dir, "weight.npy"))
edge_type_code = np.load(os.path.join(edge_dir, "edge_type_code.npy"))

TIME_DIM = 8
EDGE_TYPE_DIM = 4       
N_HEADS = 2
MAX_NEIGHBORS = 20
SEED = 42
PROJ_DIM = 32

# reply=0, mention=1, follow=2 (from build_temporal_graph.py's TYPE_CODE)
UNKNOWN_TYPE_IDX = 3     
SELF_TYPE_IDX = 4      
NUM_EDGE_TYPES = 5

def remap_edge_type_code(code):
    """-1 (unmapped, from build_temporal_edges' fillna(-1)) -> UNKNOWN_TYPE_IDX,
    everything else passes through unchanged."""
    return UNKNOWN_TYPE_IDX if code == -1 else code

In [3]:
class TimeEncode(nn.Module):
    def __init__(self, dim):
        super().__init__()
        self.dim = dim
        self.w = nn.Linear(1, dim)
        self.w.weight = nn.Parameter(
            (1.0 / 10 ** torch.linspace(0, 9, dim)).reshape(dim, 1)
        )
        self.w.bias = nn.Parameter(torch.zeros(dim))

    def forward(self, delta_t):
        delta_t = delta_t.unsqueeze(-1)
        return torch.cos(self.w(delta_t))

In [4]:
class TemporalAttentionLayer(nn.Module):
    def __init__(self, feat_dim, time_dim, n_heads,
                 num_edge_types=NUM_EDGE_TYPES, edge_type_dim=EDGE_TYPE_DIM,
                 proj_dim=PROJ_DIM):
        super().__init__()
        self.time_encode = TimeEncode(time_dim)
        self.edge_type_embed = nn.Embedding(num_edge_types, edge_type_dim)
        raw_dim = feat_dim + time_dim + edge_type_dim + 1
        self.input_proj = nn.Linear(raw_dim, proj_dim)   # NEW: raw_dim -> proj_dim, whatever raw_dim is
        self.attn = nn.MultiheadAttention(
            embed_dim=proj_dim, num_heads=n_heads, batch_first=True  # CHANGED: proj_dim, not combined_dim
        )
        self.merger = nn.Sequential(
            nn.Linear(proj_dim + feat_dim, feat_dim),   # CHANGED: proj_dim, not combined_dim
            nn.ReLU(),
        )

    def forward(self, node_feat, neighbor_feat, neighbor_time_delta,
                neighbor_edge_type, neighbor_weight, key_padding_mask):
        B = node_feat.size(0)
        device = node_feat.device

        q_time_enc = self.time_encode(torch.zeros(B, device=device))
        q_edge_type_enc = self.edge_type_embed(
            torch.full((B,), SELF_TYPE_IDX, dtype=torch.long, device=device)
        )
        q_weight = torch.ones(B, 1, device=device)
        q_raw = torch.cat([node_feat, q_time_enc, q_edge_type_enc, q_weight], dim=-1)
        q = self.input_proj(q_raw).unsqueeze(1)   # NEW: project before attention

        k_time_enc = self.time_encode(neighbor_time_delta)
        k_edge_type_enc = self.edge_type_embed(neighbor_edge_type)
        k_weight = neighbor_weight.unsqueeze(-1)
        k_raw = torch.cat([neighbor_feat, k_time_enc, k_edge_type_enc, k_weight], dim=-1)
        k = self.input_proj(k_raw)   # NEW: same projection, weights shared between q and k

        attn_out, attn_weights = self.attn(q, k, k, key_padding_mask=key_padding_mask)
        attn_out = attn_out.squeeze(1)

        merged = torch.cat([attn_out, node_feat], dim=-1)
        return self.merger(merged), attn_weights

In [5]:
class TGAT(nn.Module):
    def __init__(self, feat_dim, time_dim=TIME_DIM, n_heads=N_HEADS, num_classes=2):
        super().__init__()
        self.layer = TemporalAttentionLayer(feat_dim, time_dim, n_heads)
        self.classifier = nn.Linear(feat_dim, num_classes)

    def forward(self, node_feat, neighbor_feat, neighbor_time_delta,
                neighbor_edge_type, neighbor_weight, key_padding_mask):
        h, attn_weights = self.layer(
            node_feat, neighbor_feat, neighbor_time_delta,
            neighbor_edge_type, neighbor_weight, key_padding_mask
        )
        logits = self.classifier(h)
        return logits, attn_weights

In [6]:
def prepare_batch(node_indices, x, finder, query_time, max_neighbors=MAX_NEIGHBORS):
    B = len(node_indices)
    feat_dim = x.shape[1]
    width = max_neighbors + 1

    neighbor_feat = np.zeros((B, width, feat_dim), dtype=np.float32)
    neighbor_time_delta = np.zeros((B, width), dtype=np.float32)
    neighbor_edge_type = np.zeros((B, width), dtype=np.int64)   # NEW
    neighbor_weight = np.zeros((B, width), dtype=np.float32)    # NEW
    mask = np.ones((B, width), dtype=bool)

    for i, node in enumerate(node_indices):
        neighbor_feat[i, 0] = x[node]
        neighbor_time_delta[i, 0] = 0.0
        neighbor_edge_type[i, 0] = SELF_TYPE_IDX   # NEW
        neighbor_weight[i, 0] = 1.0                # NEW
        mask[i, 0] = False

        neighbors, edge_idxs, timestamps = finder.get_temporal_neighbors(node, query_time, max_neighbors)
        for j in range(len(neighbors)):
            neighbor_feat[i, j + 1] = x[neighbors[j]]
            neighbor_time_delta[i, j + 1] = (query_time - timestamps[j]) / 86400.0
            raw_code = finder.edge_type_code[edge_idxs[j]]                    # NEW
            neighbor_edge_type[i, j + 1] = remap_edge_type_code(int(raw_code))  # NEW
            neighbor_weight[i, j + 1] = float(finder.weight[edge_idxs[j]])      # NEW
            mask[i, j + 1] = False

    return (
        torch.tensor(x[node_indices], dtype=torch.float32),
        torch.tensor(neighbor_feat, dtype=torch.float32),
        torch.tensor(neighbor_time_delta, dtype=torch.float32),
        torch.tensor(neighbor_edge_type, dtype=torch.long),   # NEW
        torch.tensor(neighbor_weight, dtype=torch.float32),   # NEW
        torch.tensor(mask, dtype=torch.bool),
    )

In [7]:
FEATURES_PATH = "/kaggle/input/datasets/judithannbenny/tnsgat-data/sample_features.csv"
EDGES_PATH = "/kaggle/input/datasets/judithannbenny/tnsgat-data/sample_edges.csv"

In [8]:
import os

torch.manual_seed(SEED)
np.random.seed(SEED)

x = np.load(os.path.join(NODE_FEATURES_PATH, "x.npy"))
y = np.load(os.path.join(NODE_FEATURES_PATH, "y.npy"))
ids = np.load(os.path.join(NODE_FEATURES_PATH, "ids.npy"), allow_pickle=True)

src = np.load(os.path.join(TEMPORAL_EDGES_PATH, "src.npy"))
dst = np.load(os.path.join(TEMPORAL_EDGES_PATH, "dst.npy"))
timestamp = np.load(os.path.join(TEMPORAL_EDGES_PATH, "timestamp.npy"))
weight = np.load(os.path.join(TEMPORAL_EDGES_PATH, "weight.npy"))
edge_type_code = np.load(os.path.join(TEMPORAL_EDGES_PATH, "edge_type_code.npy"))

finder = NeighborFinder(len(ids), src, dst, timestamp, weight, edge_type_code)
print(f"Loaded {len(ids)} nodes, {len(src)} temporal events")

Loaded 1000 nodes, 1092 temporal events


In [9]:
query_time = int(timestamp.max()) + 1

n = len(ids)
idx = np.arange(n)
rng = np.random.RandomState(SEED)
rng.shuffle(idx)
split_point = int(0.8 * n)
train_idx, val_idx = idx[:split_point], idx[split_point:]
print(f"Train: {len(train_idx)} accounts, Val: {len(val_idx)} accounts")

Train: 800 accounts, Val: 200 accounts


In [10]:
model = TGAT(feat_dim=x.shape[1])
optimizer = torch.optim.Adam(model.parameters(), lr=1e-3)
y_tensor = torch.tensor(y, dtype=torch.long)

n_epochs = 50
for epoch in range(1, n_epochs + 1):
    model.train()
    optimizer.zero_grad()

    node_feat, neighbor_feat, neighbor_time_delta, neighbor_edge_type, neighbor_weight, mask = \
        prepare_batch(train_idx, x, finder, query_time)
    logits, _ = model(node_feat, neighbor_feat, neighbor_time_delta,
                       neighbor_edge_type, neighbor_weight, mask)
    loss = F.cross_entropy(logits, y_tensor[train_idx])
    loss.backward()
    optimizer.step()

    if epoch % 10 == 0 or epoch == 1:
        model.eval()
        with torch.no_grad():
            vnode_feat, vneighbor_feat, vneighbor_time_delta, vneighbor_edge_type, vneighbor_weight, vmask = \
                prepare_batch(val_idx, x, finder, query_time)
            val_logits, _ = model(vnode_feat, vneighbor_feat, vneighbor_time_delta,
                                   vneighbor_edge_type, vneighbor_weight, vmask)
            val_loss = F.cross_entropy(val_logits, y_tensor[val_idx])
            val_acc = (val_logits.argmax(dim=-1) == y_tensor[val_idx]).float().mean()
            train_acc = (logits.argmax(dim=-1) == y_tensor[train_idx]).float().mean()
        print(f"Epoch {epoch:3d}: train_loss={loss.item():.4f} train_acc={train_acc.item():.3f} "
              f"val_loss={val_loss.item():.4f} val_acc={val_acc.item():.3f}")

Epoch   1: train_loss=0.7154 train_acc=0.460 val_loss=0.7031 val_acc=0.550
Epoch  10: train_loss=0.5936 train_acc=0.837 val_loss=0.5893 val_acc=0.840
Epoch  20: train_loss=0.4745 train_acc=0.856 val_loss=0.4626 val_acc=0.865
Epoch  30: train_loss=0.3480 train_acc=0.869 val_loss=0.3234 val_acc=0.885
Epoch  40: train_loss=0.2977 train_acc=0.902 val_loss=0.2680 val_acc=0.910
Epoch  50: train_loss=0.2624 train_acc=0.913 val_loss=0.2564 val_acc=0.920


In [11]:
model.eval()
with torch.no_grad():
    all_idx = np.arange(n)
    node_feat, neighbor_feat, neighbor_time_delta, neighbor_edge_type, neighbor_weight, mask = \
        prepare_batch(all_idx, x, finder, query_time)
    logits, _ = model(node_feat, neighbor_feat, neighbor_time_delta,
                       neighbor_edge_type, neighbor_weight, mask)
    p_gnn = F.softmax(logits, dim=-1)[:, 1].numpy()

out = pd.DataFrame({"id": ids, "label": np.where(y == 1, "bot", "human"), "p_bot_gnn": p_gnn})
out.to_csv("/kaggle/working/tgat_predictions.csv", index=False)
print("\nSaved tgat_predictions.csv")
print(out.groupby("label")["p_bot_gnn"].mean())


Saved tgat_predictions.csv
label
bot      0.491459
human    0.105610
Name: p_bot_gnn, dtype: float32
